# Bayes Computational Project
## Group Members:
- **Benjamin Nelson**: Wrote a majority of the code relating to the SKLearn versions of each Bayes model
- **Harrison Yeboah**: Wrote the implementation for the Bernoulli Bayes model
- **Han Nguyen**: Wrote the implementation for the Gaussian Bayes model
- **Duc Le**: Worked on markdown explanations + analysis
- **Kaiden McCready**: Worked on markdown explanations + analysis 
## Project Explanation

In this project, we will be implementing naive Bayesian classifiers to the MNIST dataset, a set of handwritten digits (0-9) in 28x28px form, with assigned labels for the digit. We will be picking two numbers from the dataset (3 and 5), and having our model attempt to differentiate between the two by classifying a test set of handwritten 3's and 5's into the correct digit label. 

We will build two implementations, one assuming normally distributed parameters (using the Gaussian distribution for likelihoods) and one assuming likelihoods distrubuted with the Bernoulli distribution. Last, we will compare the results of our implementation with the plug-and-play naive Bayesian classifiers built into sklearn as a quick sanity check.

## Data

This project uses the MINST dataset, which consists of 28x28 pixel images of handwriting, and the attached labels of which digit they represent. The data is separated into a training set (with the associated images _x_ and labels _y_) and a smaller test set (similar format). For purposes of a naive Bayesian classifier, we will **flatten** the image into 28*28 = 784 individual pixels (as opposed to a 2d grid) as our parameters in our naive Baysian classifiers.

## Implementation
To start, we'll read in from our source of the MNIST dataset, separating between training and test sets.

In [70]:
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
from tabulate import tabulate
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import BernoulliNB

path = Path("mnist.npz")
if not path.exists():
    urlretrieve(
        "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz",
        path
    )

with np.load(path) as data:
    x_train = data["x_train"]
    y_train = data["y_train"]
    x_test = data["x_test"]
    y_test = data["y_test"]

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)


As expected, we have a set of 60,000 low-res, 28x28px pictures and their associated 60,000 labels. We have the same for our smaller (10,000 points) training set. Let's format the set for our project, leaving only the data points containing the numbers we want to classify (3 and 5). We'll also flatten the dataset to be 28*28 = 784 individual pixels rather than a 2D grid for each data point. We'll use uppercase X and Y to denote "formatted" data.

In [71]:
digits = [3, 5]

train_mask = np.isin(y_train, digits)
test_mask = np.isin(y_test, digits)

X_train = x_train[train_mask].reshape(-1, 784).astype(float) # flatten dataset
Y_train = y_train[train_mask]

X_test = x_test[test_mask].reshape(-1, 784).astype(float)
Y_test = y_test[test_mask]

Next, let's get the mean, standard deviation, and variance for each number type (to later compute the Baysian probabilities).

In [72]:
# within 3s, find mean and stdev
threes = X_train[Y_train == 3]

mean_3 = threes.mean(axis=0)
std_3 = threes.std(axis=0)
var_3 = threes.var(axis=0)

In [73]:
# within 5s, find mean and stdev
fives = X_train[Y_train == 5]

mean_5 = fives.mean(axis=0)
std_5 = fives.std(axis=0)
var_5 = fives.var(axis=0)

Let's also "smooth" the variance by adding an arbitrary amount to ensure we don't divide by zero when calculating the probabilities.

In [74]:
# smoothing variance

epsilon = 1e-9 * max(var_3.max(), var_5.max())
var_3_smooth = var_3 + epsilon
var_5_smooth = var_5 + epsilon

We'll get the priors (the probability of finding a 3 and a 5 in our training set). Since our dataset is exclusively made up of 3s and 5s, this should add up to 100% (1). 

In [75]:
# prior 
prior_3 = np.mean(Y_train == 3)
prior_5 = np.mean(Y_train == 5)

print(prior_3, prior_5)
print(prior_3 + prior_5)  # should be 1.0 - yay it is!

0.5307306094182825 0.46926939058171746
1.0


Here, gaussian_score finds the discriminate function of a single class by taking the log of the probability density for the normal / Gaussian distrubition for each parameter (pixel scores), then summing (instead of multiplying, because we have taken the log) across all parameters, then adding the prior, to give us our final discriminate function value for the class.  

In [76]:
def gaussian_score(x, mean, variance, prior):
    pixel_scores = -1/2 * (np.log(2 * np.pi * variance) + (x - mean) ** 2 / variance)
    return np.log(prior) + pixel_scores.sum()

From here, we can get the value of our discriminant function among both classes, then decide how to classify them for each value in our testing set.

In [77]:
# list comprehension, check all images and do predictions
predictions = np.array([
    3 if gaussian_score(x, mean_3, var_3_smooth, prior_3)
         > gaussian_score(x, mean_5, var_5_smooth, prior_5)
    else 5
    for x in X_test
])

Now, let's check how our model performs compared to the true labels on our test set.

In [78]:
labels = [3, 5]

confusion = np.array([
    [
        np.sum((Y_test == actual) & (predictions == predicted))
        for predicted in labels
    ]
    for actual in labels
])

headers = ["", "Predicted 3", "Predicted 5"]
data = [
    ["Actual 3", confusion[0, 0], f"{confusion[0, 1]}"],
    ["Actual 5", f"{confusion[1, 0]}", confusion[1, 1]]
]

print(tabulate(data, headers=headers, tablefmt="grid"))

+----------+---------------+---------------+
|          |   Predicted 3 |   Predicted 5 |
+==========+===============+===============+
| Actual 3 |           976 |            34 |
+----------+---------------+---------------+
| Actual 5 |           280 |           612 |
+----------+---------------+---------------+


In [79]:
TP = confusion[1, 1]  # actual 5, predicted 5
FP = confusion[0, 1]  # actual 3, predicted 5
FN = confusion[1, 0]  # actual 5, predicted 3

accuracy = np.trace(confusion) / confusion.sum()
recall = TP / (FN + TP)
precision = TP / (FP + TP)

f1 = 2 / (1/precision + 1/recall)

print("Accuracy:", accuracy) #np.trace gives the sum of diagonal
print("F1 Score:", f1)

Accuracy: 0.8349106203995794
F1 Score: 0.7958387516254877


[ Duc Analysis Here ]

In [80]:
# Bernoulli needs to be binarized here
X_train_bin = (X_train > 127).astype(float)
X_test_bin = (X_test > 127).astype(float)

In [81]:
# Smoothing done here
theta_3 = (X_train_bin[Y_train == 3].sum(axis=0) + 1) / (np.sum(Y_train == 3) + 2)

In [82]:
# within 5s, per-pixel probability of being "on" here
theta_5 = (X_train_bin[Y_train == 5].sum(axis=0) + 1) / (np.sum(Y_train == 5) + 2)

In [83]:
# bernoulli score function
def bernoulli_score(x, theta, prior):
    pixel_scores = x * np.log(theta) + (1 - x) * np.log(1 - theta)
    return np.log(prior) + pixel_scores.sum()

In [84]:
# Doing predictions 
bern_predictions = np.array([
    3 if bernoulli_score(x, theta_3, prior_3)
         > bernoulli_score(x, theta_5, prior_5)
    else 5
    for x in X_test_bin
])

In [85]:
# bernoulli confusion matrix
bern_confusion = np.array([
    [
        np.sum((Y_test == actual) & (bern_predictions == predicted))
        for predicted in labels
    ]
    for actual in labels
])

data = [
    ["Actual 3", bern_confusion[0, 0], bern_confusion[0, 1]],
    ["Actual 5", bern_confusion[1, 0], bern_confusion[1, 1]]
]

print(tabulate(data, headers=headers, tablefmt="grid"))

+----------+---------------+---------------+
|          |   Predicted 3 |   Predicted 5 |
+==========+===============+===============+
| Actual 3 |           936 |            74 |
+----------+---------------+---------------+
| Actual 5 |           120 |           772 |
+----------+---------------+---------------+


In [86]:
# Confusion matrix
TP = bern_confusion[1, 1]  # actual 5, predicted 5
FP = bern_confusion[0, 1]  # actual 3, predicted 5
FN = bern_confusion[1, 0]  # actual 5, predicted 3

bern_accuracy = np.trace(bern_confusion) / bern_confusion.sum()
bern_recall = TP / (FN + TP)
bern_precision = TP / (FP + TP)

bern_f1 = 2 / (1/bern_precision + 1/bern_recall)

print(bern_accuracy)
print(bern_f1)

0.8980021030494216
0.8883774453394706


Finally, we'll use the Baysian classifcation models built into sklearn to run our models rather than doing everything by hand. For each type of distribution, this function takes in the data we use to train and test with, creates predictions based on the built-in model, and returns the resulting confusion matrix.

In [90]:
#Self contained sklearn Normal(gaussian) and Bernoulli Naive Bayes

def NiaveBayesSKLearn(X_train, Y_train, X_test, Y_test, isBernoulli = False):

    #Locality rule
    frame = 0

    #Assingment of either Gaussian or Bernoulli
    if not isBernoulli:
        frame = GaussianNB()
    else:
        frame = BernoulliNB()

    #fit predict
    frame = frame.fit(X_train,Y_train)
    frame_y_predicts = frame.predict(X_test)

    #Confusion Matrix
    labels = [3, 5]
    confusion = np.array([
        [
            np.sum((Y_test == actual) & (frame_y_predicts == predicted))
            for predicted in labels
        ]
        for actual in labels
    ])

    #Making Table
    headers = ["", "Predicted " + str(2), "Predicted "  + str(5)]
    data = [
        ["Actual " + str(3), confusion[0, 0], f"{confusion[0, 1]}"],
        ["Actual " + str(5), f"{confusion[1, 0]}", confusion[1, 1]]
    ]

    accuracy = np.trace(confusion) / confusion.sum()
    recall = TP / (FN + TP)
    precision = TP / (FP + TP)

    f1 = 2 / (1/precision + 1/recall)

    #returning table
    table = tabulate(data, headers=headers, tablefmt="grid")
    return(table, accuracy, f1)

In [93]:
#Test NiaveBayesSKLearn
print("sklearn - GaussianNB")
sklearn_gaussian_results = NiaveBayesSKLearn(X_train, Y_train, X_test, Y_test, isBernoulli = False)
print(sklearn_gaussian_results[0]) # print table
print("Accuracy:", sklearn_gaussian_results[1])
print("F1:", sklearn_gaussian_results[2])
print()
print("sklearn - BernoulliNB")
sklearn_bernoulli_results = NiaveBayesSKLearn(X_train, Y_train, X_test, Y_test, isBernoulli = True)
print(sklearn_bernoulli_results[0])
print("Accuracy:", sklearn_bernoulli_results[1])
print("F1:", sklearn_bernoulli_results[2])

sklearn - GaussianNB
+----------+---------------+---------------+
|          |   Predicted 2 |   Predicted 5 |
+==========+===============+===============+
| Actual 3 |           976 |            34 |
+----------+---------------+---------------+
| Actual 5 |           280 |           612 |
+----------+---------------+---------------+
Accuracy: 0.8349106203995794
F1: 0.8883774453394706

sklearn - BernoulliNB
+----------+---------------+---------------+
|          |   Predicted 2 |   Predicted 5 |
+==========+===============+===============+
| Actual 3 |           955 |            55 |
+----------+---------------+---------------+
| Actual 5 |           150 |           742 |
+----------+---------------+---------------+
Accuracy: 0.8922187171398528
F1: 0.8883774453394706
